In [8]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [9]:
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

from sklearn.metrics import log_loss

In [10]:
train_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)

test_df = pd.read_csv(
    "/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv"
)

print(train_df.shape)
print(test_df.shape)

train_df.head()

(2000, 8)
(500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [11]:
def make_pairwise(df, train=True):

    rows = []

    for _, row in df.iterrows():

        for option in ["A","B","C","D","E"]:

            rows.append({

                "id": row["id"],

                "question": row["prompt"],

                "option": row[option],

                "option_name": option,

                "text":
                    "Question: "
                    + row["prompt"]
                    + " [SEP] Option: "
                    + row[option],

                "label":
                    int(option == row["answer"])
                    if train else -1
            })

    return pd.DataFrame(rows)


pair_train = make_pairwise(train_df)

pair_test = make_pairwise(test_df, train=False)

print(pair_train.shape)
pair_train.head()

(10000, 6)


,id,question,option,option_name,text,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,A,Question: Pick the best possible answer: What ...,0
1,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans do not e...,B,Question: Pick the best possible answer: What ...,1
2,1,Pick the best possible answer: What is Martin ...,Martin Heidegger does not believe in the exist...,C,Question: Pick the best possible answer: What ...,0
3,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that the relationshi...,D,Question: Pick the best possible answer: What ...,0
4,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that time is an illu...,E,Question: Pick the best possible answer: What ...,0


In [12]:
from sklearn.model_selection import train_test_split

tokenizer = Tokenizer(
    num_words=30000,
    oov_token="<UNK>"
)

tokenizer.fit_on_texts(pair_train["text"])

In [13]:
from sklearn.model_selection import GroupShuffleSplit

X = tokenizer.texts_to_sequences(pair_train["text"])

X = pad_sequences(
    X,
    maxlen=256,
    padding="post",
    truncating="post"
)

y = pair_train["label"].values

groups = pair_train["id"].values

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, valid_idx = next(gss.split(X, y, groups))

X_train = X[train_idx]
X_valid = X[valid_idx]

y_train = y[train_idx]
y_valid = y[valid_idx]

In [14]:
class MCQDataset(Dataset):

    def __init__(self, X, y):
        self.X = torch.LongTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(
    MCQDataset(X_train, y_train),
    batch_size=64,
    shuffle=True
)

valid_loader = DataLoader(
    MCQDataset(X_valid, y_valid),
    batch_size=64,
    shuffle=False
)

In [15]:
class BiLSTMAttention(nn.Module):

    def __init__(self, vocab_size):

        super().__init__()

        self.embedding = nn.Embedding(vocab_size, 256)

        self.lstm = nn.LSTM(
            256,
            128,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(256, 1)

        self.dropout = nn.Dropout(0.5)

        self.fc = nn.Linear(256, 1)

    def forward(self, x):

        x = self.embedding(x)

        out, _ = self.lstm(x)

        attn = torch.softmax(
            self.attention(out),
            dim=1
        )

        context = torch.sum(attn * out, dim=1)

        context = self.dropout(context)

        return self.fc(context).squeeze(1)

In [16]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = BiLSTMAttention(30000).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [17]:
epochs = 8

for epoch in range(epochs):

    model.train()

    train_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        optimizer.step()

        train_loss += loss.item()

    print(
        f"Epoch {epoch+1}/{epochs}  Loss: {train_loss/len(train_loader):.4f}"
    )

Epoch 1/8  Loss: 0.5061
Epoch 2/8  Loss: 0.4980
Epoch 3/8  Loss: 0.4627
Epoch 4/8  Loss: 0.2846
Epoch 5/8  Loss: 0.1573
Epoch 6/8  Loss: 0.0947
Epoch 7/8  Loss: 0.0601
Epoch 8/8  Loss: 0.0465


In [18]:
model.eval()

valid_scores = []

with torch.no_grad():

    for X_batch, _ in valid_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.sigmoid(outputs)

        valid_scores.extend(
            probs.cpu().numpy()
        )

valid_scores = np.array(valid_scores)

print(valid_scores[:10])

[3.9202929e-04 7.5399166e-04 3.8352667e-04 9.9678826e-01 5.0052284e-04
 2.5518544e-04 9.9734902e-01 2.2555630e-04 3.8482642e-04 2.5342248e-04]


In [19]:
# Cell 11

valid_df = pair_train.iloc[valid_idx].copy()

valid_df = valid_df.reset_index(drop=True)

valid_df["score"] = valid_scores

print(valid_df.head())
print(valid_df.shape)

   id                                           question  \
0  24  Which of the following is correct? Which of th...   
1  24  Which of the following is correct? Which of th...   
2  24  Which of the following is correct? Which of th...   
3  24  Which of the following is correct? Which of th...   
4  24  Which of the following is correct? Which of th...   

                                              option option_name  \
0  The blocking temperature of an antiferromagnet...           A   
1  The blocking temperature of an antiferromagnet...           B   
2  The blocking temperature of an antiferromagnet...           C   
3  The blocking temperature of an antiferromagnet...           D   
4  The blocking temperature of an antiferromagnet...           E   

                                                text  label     score  
0  Question: Which of the following is correct? W...      0  0.000392  
1  Question: Which of the following is correct? W...      0  0.000754  
2  Question: W

In [20]:
# Cell 12

predictions = []
true_answers = []

for _, group in valid_df.groupby("id"):

    group = group.sort_values(
        "score",
        ascending=False
    )

    predictions.append(
        group["option_name"].tolist()[:3]
    )

    true_answers.append(
        group.loc[
            group["label"] == 1,
            "option_name"
        ].values[0]
    )

print(predictions[:5])
print(true_answers[:5])

[['D', 'B', 'E'], ['B', 'D', 'A'], ['C', 'A', 'D'], ['C', 'B', 'D'], ['E', 'B', 'A']]
['D', 'B', 'C', 'C', 'E']


In [21]:
def map3(y_true, y_pred):

    score = 0.0

    for true, pred in zip(y_true, y_pred):

        if true == pred[0]:
            score += 1.0

        elif true == pred[1]:
            score += 0.5

        elif true == pred[2]:
            score += 1.0 / 3.0

    return score / len(y_true)


map_score = map3(true_answers, predictions)

print("Validation MAP@3 :", round(map_score, 5))

Validation MAP@3 : 0.98958


In [22]:
top1 = [p[0] for p in predictions]

acc = np.mean(np.array(top1) == np.array(true_answers))

print("Validation Accuracy :", round(acc,5))

Validation Accuracy : 0.98


In [23]:
test_sequences = tokenizer.texts_to_sequences(pair_test["text"])

test_sequences = pad_sequences(
    test_sequences,
    maxlen=256,
    padding="post",
    truncating="post"
)

test_loader = DataLoader(

    torch.LongTensor(test_sequences),

    batch_size=64,

    shuffle=False
)

In [24]:
model.eval()

test_scores = []

with torch.no_grad():

    for X_batch in test_loader:

        X_batch = X_batch.to(device)

        outputs = model(X_batch)

        probs = torch.sigmoid(outputs)

        test_scores.extend(probs.cpu().numpy())

pair_test["score"] = test_scores

predictions = []

ids = []

for qid, group in pair_test.groupby("id"):

    group = group.sort_values(
        "score",
        ascending=False
    )

    ids.append(qid)

    predictions.append(
        " ".join(group["option_name"].tolist()[:3])
    )

In [26]:
submission = pd.DataFrame({

    "ID": ids,

    "Prediction": predictions

})

submission.to_csv(
    "submission.csv",
    index=False
)

print(submission.head())

print("submission.csv saved")

   ID Prediction
0   1      A C E
1   2      B D E
2   3      B E D
3   4      E D C
4   5      C A D
submission.csv saved
